### Main Cleaning Code for Criteria (txt_criteria)

In [ ]:
import pandas as pd
import re
import os

def day_zero_reconstructor_v3(text, pillar_label="text"):
    """
    Final optimized cleaner for all NLP pillars.
    Protects medical terminology and purges administrative leakage.
    """
    if not isinstance(text, str) or not text.strip() or text.lower() == 'nan':
        return f"No {pillar_label} provided"

    # 1. PROTECT: Medical terms and our manual [SEP] tokens
    # We mask [SEP] so the regex cleanup doesn't treat brackets as punctuation
    text = text.replace("[SEP]", " __SEP_TOKEN__ ")

    protected_terms = ["RECIST", "CTCAE", "NYHA", "ECOG", "HbA1c", "pfs", "os", "orr"]
    for i, term in enumerate(protected_terms):
        # Case-insensitive protection for common acronyms
        text = re.sub(rf'(?i)\b{term}\b', f"__PROT_{i}__", text)

    # 2. CLEAN AACT ARTIFACTS
    text = text.replace('~', ' ')
    text = text.replace('\\>', '>')
    text = text.replace('*', ' ')
    text = re.sub(r'-{2,}', ' ', text)

    # 3. LEAKAGE PURGE (Administrative Redaction)
    admin_patterns = [
        r'(?i)\bamendment\b.*?(?=[.;:]|$)',
        r'(?i)\bprotocol\s+v(?:er|ersion)?\.?\s*[\d\.]+',
        r'(?i)\brevised\s+(?:per|on|by)\b.*?(?=[.;:]|$)',
        r'(?i)\bupdated\s+(?:as\s+of|on|protocol)\b.*?(?=[.;:]|$)',
        r'(?i)\bmodified\s+(?:on|date)\b.*?(?=[.;:]|$)'
    ]
    for pattern in admin_patterns:
        text = re.sub(pattern, ' ', text)

    # 4. RESTORE & NORMALIZE
    for i, term in enumerate(protected_terms):
        text = text.replace(f"__PROT_{i}__", term.upper())

    # Restore the [SEP] tokens
    text = text.replace("__SEP_TOKEN__", "[SEP]")

    # Final whitespace squeeze
    text = re.sub(r'\s+', ' ', text).strip()

    return text if len(text) > 5 else f"No {pillar_label} provided"


# ==============================================================================
# STANDALONE EXECUTION BLOCK
# This code ONLY runs if you execute this file directly (python text_cleaning.py)
# It is IGNORED when you import the function into your notebook.
# ==============================================================================
if __name__ == "__main__":
    # 1. Setup Paths (Relative to project root)
    input_path = 'data/project_data.csv'
    output_path = 'data/project_data_nlp_light.csv'

    if not os.path.exists(input_path):
        print(f"ERROR: Could not find {input_path}. Check your working directory.")
    else:
        # 2. Load Data
        print(f">>> Loading {input_path}...")
        df = pd.read_csv(input_path)

        # 3. Apply Sanitization
        print(">>> Executing Final Scientific Sanitization...")
        pillars = ['txt_scientific_essence', 'txt_criteria', 'txt_primary_endpoints']

        for col in pillars:
            if col in df.columns:
                label = col.replace('txt_', '')
                print(f"    Processing {col}...")
                df[col] = df[col].apply(lambda x: day_zero_reconstructor_v3(x, label))

        # 4. Save Lightweight Version for Colab
        # We keep nct_id for joining and target for stratified splitting
        cols_to_keep = ['nct_id', 'target'] + [p for p in pillars if p in df.columns]
        df_light = df[cols_to_keep].copy()

        df_light.to_csv(output_path, index=False)
        print(f"\n[SUCCESS] Sanitization complete. File saved: {output_path}")

        # 5. Verification Sample
        if 'txt_scientific_essence' in df_light.columns:
            print(f"Sample Essence: {df_light['txt_scientific_essence'].iloc[0][:100]}...")

Starting text sanitization...
Verification: Tildes remaining in cleaned column: 0
Success: Lightweight file saved as 'project_data_nlp_light.csv'

--- Sample Output ---
Original: Inclusion Criteria:~* Signed written informed consent~* Naive to focal radiation therapy in the head...
Cleaned:  Inclusion Criteria: * Signed written informed consent * Naive to focal radiation therapy in the head...


In [ ]:
import pandas as pd
import re

def deep_integrity_audit(df, original_col='txt_criteria', clean_col='txt_criteria_clean'):
    """
    Comprehensive audit to ensure cleaning is robust and scientific data is safe.
    """
    # 1. Check for Remaining Leakage Keywords
    # We look for the keywords we tried to remove
    leak_keywords = [r'(?i)\bamendment\b', r'(?i)\bprotocol\s+v(er|ersion)', r'(?i)\brevised\b']
    leak_results = {}
    for pattern in leak_keywords:
        count = df[clean_col].str.contains(pattern, regex=True, na=False).sum()
        leak_results[pattern] = count

    # 2. Check for Protected Term Integrity
    # We check if 'RECIST' exists in the same number of rows before and after
    protected_check = "RECIST"
    orig_recist = df[original_col].str.contains(protected_check, na=False).sum()
    clean_recist = df[clean_col].str.contains(protected_check, na=False).sum()

    # 3. Information Loss Analysis
    # Calculate how many characters were removed
    df['char_loss'] = df[original_col].str.len() - df[clean_col].str.len()
    # Calculate percentage loss
    df['pct_loss'] = (df['char_loss'] / df[original_col].str.len()) * 100

    extreme_loss = df[df['pct_loss'] > 40] # Rows that lost more than 40% of their text

    # --- REPORTING ---
    print("--- 1. LEAKAGE PERSISTENCE ---")
    for pattern, count in leak_results.items():
        status = "✅ CLEAN" if count == 0 else "⚠️ REMAINING"
        print(f"{pattern}: {count} occurrences ({status})")

    print("\n--- 2. SCIENTIFIC INTEGRITY ---")
    if orig_recist == clean_recist:
        print(f"✅ {protected_check} count matches: {orig_recist} rows preserved.")
    else:
        print(f"❌ {protected_check} count mismatch! Original: {orig_recist}, Clean: {clean_recist}")

    print("\n--- 3. INFORMATION LOSS ---")
    print(f"Average character loss: {df['char_loss'].mean():.1f} chars")
    print(f"Rows with >40% text removed: {len(extreme_loss)}")

    if not extreme_loss.empty:
        print("\n--- SAMPLE OF EXTREME LOSS (Check if these were just 'Summary of Changes' blocks) ---")
        idx = extreme_loss.index[0]
        print(f"ORIGINAL (First 200): {df.loc[idx, original_col][:200]}...")
        print(f"CLEANED  (First 200): {df.loc[idx, clean_col][:200]}...")

    return extreme_loss

# Execution
extreme_loss_df = deep_integrity_audit(df)

/tmp/ipykernel_113055/1419225708.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = df[clean_col].str.contains(pattern, regex=True, na=False).sum()


--- 1. LEAKAGE PERSISTENCE ---
(?i)\bamendment\b: 0 occurrences (✅ CLEAN)
(?i)\bprotocol\s+v(er|ersion): 3 occurrences (⚠️ REMAINING)
(?i)\brevised\b: 521 occurrences (⚠️ REMAINING)

--- 2. SCIENTIFIC INTEGRITY ---
❌ RECIST count mismatch! Original: 2219, Clean: 2218

--- 3. INFORMATION LOSS ---
Average character loss: 6.3 chars
Rows with >40% text removed: 0


In [ ]:
# Inspect a few of the 521 'revised' occurrences
revised_samples = df[df['txt_criteria_clean'].str.contains(r'(?i)\brevised\b', na=False)]
print("--- Inspection of Remaining 'Revised' Instances ---")
for i, text in enumerate(revised_samples['txt_criteria_clean'].head(5)):
    # Find the word 'revised' and show surrounding context
    match = re.search(r'(.{0,30})(?i)\brevised\b(.{0,30})', text)
    if match:
        print(f"Sample {i+1}: ...{match.group(0)}...")

--- Inspection of Remaining 'Revised' Instances ---
Sample 1: ...rosis (ALS), according to the revised El Escorial criteria * Subjec...
Sample 2: ...(RA) based on either the 1987-revised American College of Rheumatol...
Sample 3: ...riteria for diagnosing ALS by revised El Escorial criteria (Brooks ...
Sample 4: ...is stage I/II (defined by the revised American Society for Reproduc...
Sample 5: ...nternational prognostic score-revised (IPSS-R) of >3.5 (Intermediat...


In [ ]:
import pandas as pd
import re
import unicodedata

def definitive_strikethrough_audit(df, clean_col='txt_criteria_clean'):
    """
    Scans for every possible strikethrough encoding to ensure 100% cleanliness.
    """
    # 1. Define the 'Hit List' of all possible strike encodings
    strike_patterns = {
        'Markdown (~~)': r'~~',
        'HTML (<s>, <strike>, <del>)': r'<(s|strike|del)>',
        'AACT Tilde (~)': r'~',
        'AACT Header Dashes (---)': r'-{3,}',
        'Unicode Combining Strike (\u0334-\u0338)': r'[\u0334-\u0338]',
        'Unicode Horizontal Bar (\u2015)': r'\u2015',
        'Double Hyphen Strike (--)': r'--'
    }

    results = []
    print("--- Definitive Strikethrough Audit ---")

    for name, pattern in strike_patterns.items():
        count = df[clean_col].str.contains(pattern, regex=True, na=False).sum()
        status = "✅ GONE" if count == 0 else f"⚠️ {count} REMAINING"
        results.append({'Encoding': name, 'Status': status})

    audit_df = pd.DataFrame(results)
    print(audit_df)

    # 2. Character-Level Scan (The 'Invisible' Check)
    # We check every character in the first 5000 rows for 'Combining' marks
    print("\n--- Invisible Character Scan ---")
    sample_text = "".join(df[clean_col].fillna("").head(5000).tolist())
    combining_marks = [c for c in sample_text if unicodedata.combining(c)]

    if not combining_marks:
        print("✅ No invisible combining characters (strikethrough overlays) detected.")
    else:
        print(f"⚠️ Detected {len(set(combining_marks))} types of combining marks: {set(combining_marks)}")

    # 3. Visual Verification of 'Most Changed' Rows
    # These are the rows where the cleaner did the most work.
    df['diff'] = df['txt_criteria'].str.len() - df[clean_col].str.len()
    top_changed = df.sort_values(by='diff', ascending=False).head(3)

    print("\n--- Visual Verification (Top 3 Most Cleaned Rows) ---")
    for i, (idx, row) in enumerate(top_changed.iterrows()):
        print(f"\n[Sample {i+1}] Index: {idx} | Chars Removed: {int(row['diff'])}")
        print(f"CLEANED: {row[clean_col][:250]}...")

    return audit_df

# Execution
audit_results = definitive_strikethrough_audit(df)

--- Definitive Strikethrough Audit ---


/tmp/ipykernel_113055/310827552.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = df[clean_col].str.contains(pattern, regex=True, na=False).sum()


                         Encoding  Status
0                   Markdown (~~)  ✅ GONE
1     HTML (<s>, <strike>, <del>)  ✅ GONE
2                  AACT Tilde (~)  ✅ GONE
3        AACT Header Dashes (---)  ✅ GONE
4  Unicode Combining Strike (̴-̸)  ✅ GONE
5      Unicode Horizontal Bar (―)  ✅ GONE
6       Double Hyphen Strike (--)  ✅ GONE

--- Invisible Character Scan ---
⚠️ Detected 2 types of combining marks: {'́', 'ͯ'}

--- Visual Verification (Top 3 Most Cleaned Rows) ---

[Sample 1] Index: 16604 | Chars Removed: 1122
CLEANED: Inclusion Criteria: * Receiving dialysis for ESRD for ≥3 months. Incident dialysis participants (under ; incident dialysis participants must be on ESA for ≥ 4 weeks prior to screening. * Mean of the 3 most recent central lab Hb values during the Scre...

[Sample 2] Index: 28794 | Chars Removed: 1081
CLEANED: Inclusion Criteria: * Ability to provide written informed consent (parental/guardian consent and participant assent if \<18 years of age) * Age ≥6 years * Bod

### Biobert

In [1]:
# 1. Install Transformers
!pip install transformers -q

In [1]:
!pip install gdown -q

In [2]:

import gdown

# This is the ID from your Google Drive link
file_id = '1hX8LnHQSNX6JnTb_dWrIf614tgoIcNF_'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'project_data_nlp_light.csv'

# This moves the file into the GPU's local folder
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1hX8LnHQSNX6JnTb_dWrIf614tgoIcNF_
From (redirected): https://drive.google.com/uc?id=1hX8LnHQSNX6JnTb_dWrIf614tgoIcNF_&confirm=t&uuid=ed2bd535-f0d2-44c2-8061-f64c198c202b
To: /content/project_data_nlp_light.csv
100%|██████████| 163M/163M [00:01<00:00, 87.0MB/s] 


'project_data_nlp_light.csv'

In [ ]:
import os

# 1. Define your project folder name
PROJECT_FOLDER = "LeWagonProject" # Change this to your actual folder name
BASE_PATH = f"/content/drive/MyDrive/{PROJECT_FOLDER}"

# 2. Check if the folder exists, if not, create it
if not os.path.exists(BASE_PATH):
    print(f"Path {BASE_PATH} not found. Creating it...")
    os.makedirs(BASE_PATH)
    print("Folder created successfully.")
else:
    print(f"Path {BASE_PATH} already exists. Ready to save.")

# 3. Define the final file path for your embeddings
SAVE_PATH = os.path.join(BASE_PATH, "biobert_embeddings_raw.npy")
print(f"Embeddings will be saved to: {SAVE_PATH}")

In [2]:
!ls /content/drive/MyDrive/

ls: cannot access '/content/drive/MyDrive/': No such file or directory


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from google.colab import drive

# 1. Mount Drive for persistent storage
drive.mount('/content/drive')

# 2. Load Data
df = pd.read_csv('project_data_nlp_light.csv')

# 3. CRITICAL: Data Type Enforcement
# This prevents the tokenizer from crashing on empty/NaN values
df['txt_criteria_clean'] = df['txt_criteria_clean'].astype(str)

# 4. Dataset Class
class ClinicalCriteriaDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx]

# 5. Initialize Model and Tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")
model = AutoModel.from_pretrained("dmis-lab/biobert-v1.1").to(device)

# 6. Inference Function (as defined previously)
def get_embeddings(dataloader):
    model.eval()
    all_embeddings = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Computing Embeddings"):
            inputs = tokenizer(batch, padding=True, truncation=True,
                               max_length=512, return_tensors="pt").to(device)
            outputs = model(**inputs)

            # Mean Pooling logic
            last_hidden_state = outputs.last_hidden_state
            attention_mask = inputs['attention_mask']
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
            sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            mean_embeddings = sum_embeddings / sum_mask

            all_embeddings.append(mean_embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

# 7. SMOKE TEST (Run this first to verify)
print("Starting Smoke Test...")
test_dataset = ClinicalCriteriaDataset(df['txt_criteria_clean'].iloc[:100].tolist())
test_loader = DataLoader(test_dataset, batch_size=10)
test_embeddings = get_embeddings(test_loader)
print(f"Smoke test successful. Shape: {test_embeddings.shape}")

# 8. FULL RUN (Only if Smoke Test passes)
# Uncomment the lines below to run the full dataset
# full_dataset = ClinicalCriteriaDataset(df['txt_criteria_clean'].tolist())
# full_loader = DataLoader(full_dataset, batch_size=32)
# final_embeddings = get_embeddings(full_loader)
# np.save('/content/drive/MyDrive/biobert_embeddings_raw.npy', final_embeddings)